In [1]:
import os
import json
import asyncio
import openai               
from dotenv import load_dotenv 
import pandas as pd  
load_dotenv()

True

In [2]:
category_schemas = json.load(open('../config/sdoh_extraction_schema_strict_type_simple_experiencer.json', 'r'))

In [3]:
client = openai.OpenAI()


In [4]:
from glob import glob

In [6]:
all_documents = {}

# Use os.path to handle cross-platform file paths cleanly
for file_path in glob("../data/processed_data/*.json"):
    # os.path.basename gets "file.json", os.path.splitext splits it into ("file", ".json")
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    document_data = []
    for k, v in data.items():
        sentence_data = {}
        sentence_data['sentence'] = v['sentence']
        tmp = []
        if "SDoH" in v:
            for i in v["SDoH"]:
                tmp.append(list(i.keys())[0])
        sentence_data['categories'] = tmp
        document_data.append(sentence_data)
        
    all_documents[file_name] = document_data

In [7]:
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

context_window_size = 3

def process_document(doc_sentences, model="gpt-4o-2024-08-06"):
    """Processes a single document chronologically to maintain the context window."""
    
    for i, item in enumerate(doc_sentences):
        item["extracted_predictions"] = {}
        current_categories = item.get("categories", [])
        
        if not current_categories:
            continue

        unique_categories = list(dict.fromkeys(current_categories))
        
        # Build the context string from previous sentences
        start_index = max(0, i - context_window_size)
        previous_sentences = [
            doc.get("sentence", "")
            for doc in doc_sentences[start_index:i]
        ]
        previous_context = " ".join(previous_sentences)
        
        for category in unique_categories:

            if category not in category_schemas:
                item["extracted_predictions"][category] = {
                    "error": f"Category '{category}' not found in schema config."
                }
                continue

            func_schema = category_schemas[category]

            tool_schema = {
                "type": "function",
                "function": func_schema
            }
            
            prompt = f"""Context sentences for reference: "{previous_context}"

Current Sentence to process: "{item.get('sentence', '')}"

Target Category: {category}

Task:
Extract structured information strictly for the Current Sentence.

Rules:
1. Use the Context sentences ONLY to resolve pronouns or identify the missing Experiencer.
2. Do not extract events that only occurred in the Context.
3. Return the JSON required by the {category} schema.
4. For enum fields, only use values defined in the schema.
5. If no valid information for this category appears in the Current Sentence, return:
   {{"extracted_conditions": []}}
"""

            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "You are a clinical SDoH information extraction assistant. "
                                "You must call the provided tool and follow the schema exactly. "
                                "Never return values outside the enum."
                            )
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    tools=[tool_schema],
                    tool_choice={
                        "type": "function",
                        "function": {
                            "name": func_schema["name"]
                        }
                    },
                    temperature=1
                )

                tool_calls = response.choices[0].message.tool_calls

                if not tool_calls:
                    item["extracted_predictions"][category] = {
                        "error": "No tool call returned"
                    }
                    continue

                parsed_result_str = tool_calls[0].function.arguments
                
                try:
                    parsed_result = json.loads(parsed_result_str)
                except json.JSONDecodeError:
                    parsed_result = {
                        "error": "JSON parsing failed",
                        "raw_output": parsed_result_str
                    }
                    
                item["extracted_predictions"][category] = parsed_result

            except Exception as e:
                item["extracted_predictions"][category] = {
                    "error": f"API Error: {str(e)}"
                }
                
    return doc_sentences

In [ ]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Set up output directory
output_dir = "../output/gpt4o_type_simple_experiencer"
os.makedirs(output_dir, exist_ok=True)

max_concurrent_docs = 4 

with ThreadPoolExecutor(max_workers=max_concurrent_docs) as executor:
    # Map the executing function to the corresponding file_name
    futures = {
        executor.submit(process_document, doc_data, model="gpt-4o-2024-08-06"): file_name 
        for file_name, doc_data in all_documents.items()
    }
    
    for future in tqdm(as_completed(futures), total=len(all_documents), desc="Processing Documents"):
        # Retrieve the original file_name for this specific future
        file_name = futures[future]
        
        try:
            # Get the completed extractions
            processed_doc = future.result()
            
            # Save using the original file name
            output_filename = os.path.join(output_dir, f"{file_name}_extracted.json")
            
            with open(output_filename, "w", encoding="utf-8") as f:
                json.dump(processed_doc, f, indent=4, ensure_ascii=False)
                
        except Exception as e:
            print(f"Document '{file_name}' failed to process: {e}")

print(f"\nAll documents have been successfully processed and saved in '{output_dir}'.")

Processing Documents:  74%|██████████████▊     | 23/31 [09:58<03:12, 24.09s/it]